In [8]:
from transformers import AutoModelForMaskedLM, AutoTokenizer, AutoModelForCausalLM
from interpreto import ModelWithSplitPoints
from datasets import load_dataset

# model = AutoModelForMaskedLM.from_pretrained("EuroBERT/EuroBERT-210m", trust_remote_code=True)
# tokenizer = AutoTokenizer.from_pretrained("EuroBERT/EuroBERT-210m")
model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")
print(model)
# split_point = "model.layers.1.mlp"
split_point = ["transformer.h.1.mlp.c_proj", "transformer.h.2.mlp.c_proj", "transformer.h.11.mlp.c_proj", "transformer.ln_f"]
splitted_model = ModelWithSplitPoints(
    model_or_repo_id=model,
    tokenizer=tokenizer,
    split_points=split_point,
    device_map="cuda",
    batch_size=10,
)
# split_point = ["model.layers.1.mlp", "model.layers.2.mlp", "model.layers.3.mlp", 
#               "model.layers.4.mlp", "model.layers.5.mlp", "model.layers.6.mlp",
#               "model.layers.7.mlp", "model.layers.8.mlp", "model.layers.9.mlp",
#               "model.layers.10.mlp"]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


In [9]:
rotten_tomatoes = load_dataset("cornell-movie-review-data/rotten_tomatoes")["train"]["text"][:4]
print(rotten_tomatoes)

['the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson\'s expanded vision of j . r . r . tolkien\'s middle-earth .', 'effective but too-tepid biopic', 'if you sometimes like to go to the movies to have fun , wasabi is a good place to start .']


In [10]:
from logit_lens import LogitLens

logit_lens = LogitLens(splitted_model, tokenizer, normalization = True, nb_token = 6)
logit_lens.lens(rotten_tomatoes)

No head name specified, trying to find a suitable head in the model.
Tokenizer does not have a padding token. Setting a default padding token.
torch.Size([4, 53])
Producing lens prediction for all activations.
torch.Size([2, 53, 768])
torch.Size([4, 53])
torch.Size([2, 53, 768])


IndexError: index 2 is out of bounds for axis 0 with size 2